# Alita v3 Model Preparation for TorchServe

This notebook prepares the Alita v3 model for serving by:
1. Converting class names to TorchServe format
2. Ensuring the compiled model is ready for serving

In [1]:
import json
import os
from pathlib import Path

## Convert class names to TorchServe format

In [2]:
# Load original class names
with open('original-model/Exp_60_run_01_class_names.json', 'r') as f:
    class_names = json.load(f)

print(f"Original class names: {len(class_names)} classes")
print(f"First 5 classes: {class_names[:5]}")

Original class names: 81 classes
First 5 classes: ['banded_dotterel', 'banded_rail', 'bellbird', 'black_backed_gull', 'black_billed_gull']


In [3]:
# Convert to TorchServe format (index -> name mapping)
index_to_name = {str(i): name for i, name in enumerate(class_names)}

print(f"Converted format sample:")
for i in range(min(5, len(index_to_name))):
    print(f"  {i}: {index_to_name[str(i)]}")

Converted format sample:
  0: banded_dotterel
  1: banded_rail
  2: bellbird
  3: black_backed_gull
  4: black_billed_gull


In [4]:
# Ensure exported-model directory exists
os.makedirs('exported-model', exist_ok=True)

# Save to TorchServe format
with open('exported-model/index_to_name.json', 'w') as f:
    json.dump(index_to_name, f, indent=2)

print(f"Saved index_to_name.json with {len(index_to_name)} classes")

Saved index_to_name.json with 81 classes


## Verify compiled model exists

In [5]:
# Check if compiled model exists
compiled_model_path = 'exported-model/alitav3_compiled_cpu.pt'
if os.path.exists(compiled_model_path):
    print(f"✓ Compiled model found at {compiled_model_path}")
    
    # Get file size
    size_mb = os.path.getsize(compiled_model_path) / (1024 * 1024)
    print(f"  Model size: {size_mb:.1f} MB")
else:
    print(f"✗ Compiled model not found at {compiled_model_path}")
    print("  Please run the alitav3_compile.ipynb notebook first")

✓ Compiled model found at exported-model/alitav3_compiled_cpu.pt
  Model size: 81.3 MB


## Summary

Files prepared for TorchServe:
- `exported-model/index_to_name.json` - Class name mapping
- `exported-model/alitav3_compiled_cpu.pt` - Compiled model (if exists)

Next steps:
1. Create the handler file
2. Run torch-model-archiver to create the .mar file
3. Build and test the Docker container